# ROGII Neighbor Profile Transfer: Honest 773-Well Audit

A recent ROGII Discussion asked whether the signal below the per-well line oracle comes from copying the full interpreted TVT shape of nearby wells rather than transferring only a smooth structural surface. This notebook tests the simplest legal version of that idea with leave-one-well-out validation on all **773 training wells** and **3,783,989 evaluation rows**.

Result: nearest-well TVT shape transfer contains a small real signal, but simple normalized-MD copying is nowhere near the public 7.x pipelines. The best fixed configuration improved the last-visible-TVT anchor from about **15.9099** to **15.6153** pooled RMSE. We did **not** spend a leaderboard submission on it.

## Method

For each held-out target well, only other training wells are eligible donors. The evaluation suffix is mapped to a normalized measured-depth coordinate

$$q_i=\frac{MD_i-MD_{start}}{MD_{end}-MD_{start}}.$$

Each donor contributes its TVT change relative to its own last visible boundary,

$$\Delta T_d(q)=T_d(q)-T_{d,start}.$$

The target transfer is anchored to the target's own last visible TVT,

$$\hat T^{NN}_w(q)=T_{w,start}+\sum_d \omega_d\Delta T_d(q),$$

where donor weights are inverse squared median trajectory distances. Opposite drilling directions are rejected. The transferred curve is finally shrunk toward the last-visible-TVT anchor:

$$\hat T_w=(1-\alpha)T_{w,start}+\alpha\hat T^{NN}_w.$$

The distance threshold, donor count, and shrinkage are fixed globally. No held-out suffix truth is used for per-well selection.

In [ ]:
import pandas as pd

results = pd.DataFrame([
    {"route": "last-visible TVT anchor", "distance_ft": 0, "max_donors": 0, "blend": 0.00, "coverage": 0.0000, "pooled_rmse": 15.9099, "p95_rmse": 29.0108},
    {"route": "very-close transfer", "distance_ft": 150, "max_donors": 3, "blend": 0.75, "coverage": 0.0155, "pooled_rmse": 15.8578, "p95_rmse": 29.0108},
    {"route": "close transfer", "distance_ft": 300, "max_donors": 3, "blend": 0.25, "coverage": 0.1371, "pooled_rmse": 15.8974, "p95_rmse": 28.7394},
    {"route": "mid-range transfer", "distance_ft": 600, "max_donors": 3, "blend": 0.30, "coverage": 0.6158, "pooled_rmse": 15.6207, "p95_rmse": 27.6376},
    {"route": "best fixed route", "distance_ft": 750, "max_donors": 5, "blend": 0.30, "coverage": 0.6986, "pooled_rmse": 15.6153, "p95_rmse": 28.1967},
    {"route": "wider transfer", "distance_ft": 900, "max_donors": 5, "blend": 0.30, "coverage": 0.7542, "pooled_rmse": 15.6470, "p95_rmse": 28.1967},
])
results["gain_vs_anchor"] = 15.9099 - results["pooled_rmse"]
results

## What the grid says

The strict `<150 ft` rule covers only **12 of 773 wells** in this trajectory-distance definition. Expanding the radius increases coverage, but full-strength copying quickly becomes harmful. The stable region is broad shrinkage: about 20–30% donor shape at 600–900 ft.

Averaging up to five donors is slightly better than taking only the nearest donor. This suggests that the transferable component is a low-amplitude regional shape prior, not an exact copy-paste solution.

In [ ]:
import matplotlib.pyplot as plt

plot = results[results["distance_ft"] > 0].copy()
fig, ax = plt.subplots(figsize=(9, 4.5), dpi=140)
scatter = ax.scatter(
    100 * plot["coverage"], plot["pooled_rmse"],
    s=80 + 500 * plot["blend"], c=plot["distance_ft"], cmap="viridis"
)
for row in plot.itertuples():
    ax.annotate(f"{row.distance_ft:.0f} ft / a={row.blend:.2f}",
                (100 * row.coverage, row.pooled_rmse), xytext=(5, 5),
                textcoords="offset points", fontsize=8)
ax.axhline(15.9099, color="crimson", linestyle="--", linewidth=1.2,
           label="last-visible anchor")
ax.set_xlabel("wells receiving a donor shape (%)")
ax.set_ylabel("pooled RMSE (ft, lower is better)")
ax.set_title("Neighbor shape transfer: coverage helps only with strong shrinkage")
ax.grid(alpha=0.2)
ax.legend()
fig.colorbar(scatter, ax=ax, label="distance threshold (ft)")
plt.tight_layout()
plt.show()

## Why the simple method stalls

Normalized MD assumes that geological events occur at the same relative progress along two wells. That is usually false. Two nearby wells may have different lengths, drilling speeds, azimuths, and contact locations. A copied TVT wiggle can therefore arrive at the wrong MD even when its shape is locally useful.

The experiment also does not align donor and target GR sequences. It transfers a curve because the geometry is nearby, not because the observed rocks confirm the same stratigraphic phase. That is why strong shrinkage is load-bearing.

## Agent-ready follow-ups

A stronger neighbor route should keep the honest leave-one-well-out contract and change the coordinate system:

1. align donor and target in stratigraphic phase using bounded DTW, NCC, PF, or HMM;
2. separate wells by drilling direction before donor search;
3. compare path-to-path distance, typewell similarity, and formation-frame similarity;
4. learn only a bounded residual around a strong public/physical trajectory;
5. use visible-prefix backtests to gate a donor, never hidden suffix truth;
6. report field-grouped CV in addition to ordinary grouped-by-well CV.

The important negative result is not that neighbors are useless. It is that **raw normalized-MD copy-paste cannot explain sub-7 performance**.

## Attribution and reproducibility

The experiment was motivated by the Kaggle Discussion *Where does the top-team signal come from below the per-well line-oracle?* and comments proposing full-curve transfer for nearby wells. Those posts are hypotheses, not proof; all numeric results above come from our independent 773-well implementation.

The full reusable script is published in the GitHub repository as `scripts/run_neighbor_profile_cv.py`. This notebook contains only aggregate, non-sensitive results and runs deterministically without competition data.